# 競艇予測モデル — Colab 実行ドライバ

**このノートブックは成果物ではありません。** 実装は `src/kyotei/` の再実行可能なスクリプトにあり、ここはそれを Colab から叩くだけの薄いドライバです（SPEC §5）。ロジックをこのノートに書かないでください。

## GPU について

SPEC §4 は PyTorch / ニューラルネットを禁じ、LightGBM (GBDT) を指定しています。**GBDT は CPU 律速なので、GPU ランタイムを選んでも速くなりません。** ランタイムは「CPU・ハイメモリ」を選んでください。

Colab を使う本当の利点はこちらです:

| 課題 | Colab での解決 |
|---|---|
| 取得に約2.4時間かかる（サーバのTTFBが約10秒） | Drive に保存し、セッションが切れても再取得しない |
| コンテナが揮発してデータが消える | `data/` を Drive に置いて永続化 |
| 学習の反復 | メモリの大きいランタイムで全期間を一度に載せる |

## 手順

上から順に実行します。取得は**冪等**なので、途中で切れても再実行すれば続きから進みます。

## 1. Drive をマウントしてデータを永続化する

`data/` を Drive 上に置き、リポジトリからシンボリックリンクします。これで**取得済みのLZHがセッションを越えて残ります**。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
PERSIST = pathlib.Path('/content/drive/MyDrive/kyotei-predict')
(PERSIST / 'data').mkdir(parents=True, exist_ok=True)
(PERSIST / 'reports').mkdir(parents=True, exist_ok=True)
print('persisting to', PERSIST)

## 2. リポジトリと依存関係

`REPO_URL` を自分のリモートに置き換えてください。

In [ ]:
REPO_URL = 'https://github.com/Daccho/kyotei-predict.git'
BRANCH = 'claude/boat-racing-model-mqgqbn'

import os, pathlib, subprocess

REPO = pathlib.Path('/content/kyotei-predict')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'pull', '--ff-only'], check=False)

# data/ and reports/ live on Drive so nothing is re-downloaded after a restart.
for name in ('data', 'reports'):
    local = REPO / name
    if local.is_symlink() or local.exists() and not local.is_dir():
        local.unlink()
    elif local.is_dir() and not any(local.iterdir()):
        local.rmdir()
    if not local.exists():
        local.symlink_to(PERSIST / name)
    print(name, '->', local.resolve())

In [ ]:
# lhafile decodes LZH in pure Python, so no apt-get lhasa is needed.
!pip -q install 'polars>=1.0' 'lhafile>=0.3' 'lightgbm>=4.3' 'scikit-learn>=1.4' pyarrow matplotlib requests pytest
import sys
sys.path.insert(0, '/content/kyotei-predict/src')
import kyotei; print('kyotei ok')

## 3. テストを通す

**先にこれを実行してください。** リーク検証テスト（未来のレコードを改変しても過去の特徴量が変わらないこと）と、確率の合計が 1.0 になる検証がここに入っています。落ちている状態で先に進む意味はありません。

In [ ]:
!cd /content/kyotei-predict && python -m pytest tests/ -q --ignore=tests/test_schema.py

## 4. データ取得（冪等・全体1req/s上限）

オリジンの TTFB が約10秒あるため、待ち時間を12ワーカーで隠しつつ、**トークンバケットで全体を1req/s以下**に保っています。ワーカー数を増やしてもリクエスト頻度は上がりません（`--rate` が上限）。

全期間（2015–現在、約8450ファイル）で**約2.4時間**。Colab のセッション上限に当たったら、このセルをもう一度実行すれば取得済み分はスキップされて続きから進みます。

まず短い期間で試すなら `--end` を縮めてください。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.download --dry-run --start 2015-01-01 --end 2026-12-31 --fan

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.download --start 2015-01-01 --end 2026-12-31 --kind both --fan

## 5. パースして parquet 化（DB不要）

**年ごとのパース成功率**が出ます。特定の年だけ成功率が落ちていたらレイアウト変更のサインなので、先に原因を調べてください（SPEC §3.2）。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.export --start 2015-01-01 --end 2026-12-31

## 6. 特徴量（リーク防止）

過去成績はすべて「そのレースの発走前」のみから計算されます。累積和から現在行を引く形なので、未来を含む窓が構造的に存在しません。

In [ ]:
import polars as pl
from kyotei import features as ft

raw = pl.read_parquet('data/parquet/entries.parquet')
print('entries:', raw.height)

built = ft.build(raw)
assert built.height == raw.height, f'row count changed: {raw.height} -> {built.height}'
built.write_parquet('data/parquet/features.parquet')
print('features:', built.height, 'rows,', len(built.columns), 'columns')
print('morning feature count:', len(ft.MORNING_FEATURES))

## 7. 学習

`--feature-set morning` が**実際に賭けられる**設定です（番組表のみ）。`prerace` は直前情報を、`realised` は実際の進入コースを足しますが、どちらも締切前には手に入らないので**診断用の上限値**としてしか使えません。両方回して差を見ると、進入コースがどれだけ効いているかが分かります。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.model --feature-set morning --lane1-baseline --walk-forward

In [ ]:
# Diagnostic ceiling: how much the realised course is worth. NOT bettable.
!cd /content/kyotei-predict && python -m kyotei.model --feature-set realised \
    --model-out data/parquet/model_realised.txt

## 8. バックテスト（検証期間 2024）

閾値のチューニングは**必ず valid（2024年）で**行ってください。test（2025–2026）は最後に一度だけです。

回収率が 100% を超えたら、成功ではなくまずリークを疑ってください（SPEC §8）。公開されている競艇AIは概ね 80〜95% です。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.backtest --split valid \
    --payouts data/parquet/payouts.parquet

## 9. test 期間の最終評価 — **一度だけ**

`--final` を付けないと実行を拒否します。閾値を valid で決め切ってから、このセルを**1回だけ**回してください。何度も回して閾値を選び直したら、test はもう test ではありません。

In [ ]:
# !cd /content/kyotei-predict && python -m kyotei.backtest --split test --final \
#     --payouts data/parquet/payouts.parquet

## 10. 当日の買い目レポート

`reports/YYYY-MM-DD.md` に出力されます。Drive 上なのでそのまま残ります。

In [ ]:
!cd /content/kyotei-predict && python -m kyotei.predict \
    --payouts data/parquet/payouts.parquet --threshold 1.20

import datetime, pathlib
from IPython.display import Markdown, display

today = pathlib.Path(f'reports/{datetime.date.today():%Y-%m-%d}.md')
display(Markdown(today.read_text(encoding='utf-8') if today.exists() else '未生成'))